# UAVDT Notebook 02B — Visual Metric Homography Calibration

This notebook replaces the rough automatic BEV homography with a visually calibrated road-plane homography.

Goal:

```text
reference UAV frame + selected road/lane points
→ image-to-ground metric homography
→ checked BEV road view
→ saved calibration JSON for later notebooks
```

The notebook supports two modes:

1. **Preset points** — edit point coordinates in a Python list.
2. **Interactive click calibration** — click points on the image using matplotlib in Colab.

The output is saved to Google Drive and can be used by Notebook 03A/04 to project vehicles directly into metric road coordinates.


In [ ]:
#@title 1. Set local project paths
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from notebook_local import resolve_project_dir, print_local_setup

PROJECT_DIR = resolve_project_dir()
WORK_DIR = PROJECT_DIR / 'work'
SAMPLE_DIR = WORK_DIR / 'M1401_sample' / 'images'
NB02B_DIR = WORK_DIR / 'notebook_02b_visual_metric_homography'
NB02B_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('SAMPLE_DIR:', SAMPLE_DIR)
print('NB02B_DIR:', NB02B_DIR)
print('Sample images exist:', SAMPLE_DIR.exists())
print('Number of sample jpg images:', len(list(SAMPLE_DIR.glob('*.jpg'))) if SAMPLE_DIR.exists() else 0)


In [ ]:
#@title 2. Choose reference frame
REFERENCE_FRAME_INDEX = 0 #@param {type:'integer'}

sample_images = sorted(list(SAMPLE_DIR.glob('*.jpg')) + list(SAMPLE_DIR.glob('*.png')))
if len(sample_images) == 0:
    raise FileNotFoundError('No sample images found in ' + str(SAMPLE_DIR))

if REFERENCE_FRAME_INDEX < 0 or REFERENCE_FRAME_INDEX >= len(sample_images):
    raise ValueError('REFERENCE_FRAME_INDEX is out of range.')

ref_path = sample_images[REFERENCE_FRAME_INDEX]
img_bgr = cv2.imread(str(ref_path))
if img_bgr is None:
    raise RuntimeError('Could not read image: ' + str(ref_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
IMG_H, IMG_W = img_rgb.shape[:2]

print('Reference frame:', ref_path)
print('Image size:', IMG_W, 'x', IMG_H)

plt.figure(figsize=(14, 7))
plt.imshow(img_rgb)
plt.title('Reference frame: ' + ref_path.name)
plt.axis('off')
plt.show()


## 3. Calibration concept

We calibrate a homography from **image pixels** to **ground-plane metric coordinates**.

Use points that lie on the road plane, ideally lane-line intersections, lane boundaries, crosswalk corners, or road-edge points.

Coordinate convention:

```text
X meters: across the road, left-to-right in the BEV map
Y meters: along the road, near-to-far from the camera
Z meters: up from road plane
```

You need at least 4 point pairs, but 6–12 points are better because `cv2.findHomography` can fit a more stable transform.


In [ ]:
#@title 3A. Define calibration mode and metric target layout
CALIBRATION_MODE = 'preset_points' #@param ['preset_points', 'interactive_click_points']

# Rough metric dimensions for the visible road patch.
# These are not required to be perfect, but should be plausible.
# If the road has around 6 lanes at ~3.5 m each, width may be ~21 m.
ROAD_WIDTH_M = 24.0 #@param {type:'number'}
ROAD_LENGTH_M = 90.0 #@param {type:'number'}

# Output BEV preview resolution. This is only for visualization.
BEV_PREVIEW_WIDTH_PX = 800 #@param {type:'integer'}
BEV_PREVIEW_HEIGHT_PX = 1200 #@param {type:'integer'}

print('CALIBRATION_MODE:', CALIBRATION_MODE)
print('ROAD_WIDTH_M:', ROAD_WIDTH_M)
print('ROAD_LENGTH_M:', ROAD_LENGTH_M)
print('BEV preview size:', BEV_PREVIEW_WIDTH_PX, 'x', BEV_PREVIEW_HEIGHT_PX)


In [ ]:
#@title 3B. Preset calibration points
# Edit these points after looking at the reference frame.
# Each image point [u, v] must correspond to the same-index ground point [x_m, y_m].
# Use points on the road plane only.

# The default values are rough starting points for the M1401-like frame.
# You should adjust them visually.
image_points_px = np.array([
    [260, 470],   # near-left road/lane point
    [1010, 470],  # near-right road/lane point
    [410, 160],   # far-left road/lane point
    [720, 160],   # far-right road/lane point
    [425, 330],   # mid-left lane point
    [875, 330],   # mid-right lane point
], dtype=np.float32)

ground_points_m = np.array([
    [0.0, 0.0],
    [ROAD_WIDTH_M, 0.0],
    [0.0, ROAD_LENGTH_M],
    [ROAD_WIDTH_M, ROAD_LENGTH_M],
    [0.0, ROAD_LENGTH_M * 0.45],
    [ROAD_WIDTH_M, ROAD_LENGTH_M * 0.45],
], dtype=np.float32)

print('Preset image_points_px:')
print(image_points_px)
print('Preset ground_points_m:')
print(ground_points_m)


In [ ]:
#@title 3C. Optional interactive click calibration
# This cell is optional. Run it only if CALIBRATION_MODE is interactive_click_points.
# In Colab, matplotlib clicking can be inconsistent. If it does not work,
# use the preset point list above and edit coordinates manually.

if CALIBRATION_MODE == 'interactive_click_points':
    print('Interactive mode selected.')
    print('Click points in this order:')
    print('  1 near-left, 2 near-right, 3 far-left, 4 far-right')
    print('Optionally click more road/lane points, then press Enter.')
    
    plt.figure(figsize=(14, 7))
    plt.imshow(img_rgb)
    plt.title('Click calibration points, then press Enter')
    plt.axis('on')
    clicked = plt.ginput(n=-1, timeout=0)
    plt.close()
    
    if len(clicked) < 4:
        raise RuntimeError('Need at least 4 clicked points.')
    
    image_points_px = np.array(clicked, dtype=np.float32)
    print('Clicked image points:')
    print(image_points_px)
    
    # For interactive mode, build a simple target layout.
    # First four clicked points are interpreted as near-left, near-right, far-left, far-right.
    base_targets = [
        [0.0, 0.0],
        [ROAD_WIDTH_M, 0.0],
        [0.0, ROAD_LENGTH_M],
        [ROAD_WIDTH_M, ROAD_LENGTH_M],
    ]
    
    if len(clicked) == 4:
        ground_points_m = np.array(base_targets, dtype=np.float32)
    else:
        print('More than 4 points were clicked.')
        print('Edit ground_points_m manually in the next cell if you need exact coordinates.')
        extra = []
        for k in range(len(clicked) - 4):
            frac = (k + 1) / (len(clicked) - 3)
            extra.append([ROAD_WIDTH_M * 0.5, ROAD_LENGTH_M * frac])
        ground_points_m = np.array(base_targets + extra, dtype=np.float32)
else:
    print('Preset mode selected. Skipping interactive clicking.')


In [ ]:
#@title 4. Visual check of selected calibration points
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(img_rgb)
ax.set_title('Calibration image points')
ax.axis('on')

for i, (u, v) in enumerate(image_points_px):
    ax.scatter([u], [v], s=80)
    ax.text(u + 8, v - 8, str(i), fontsize=14, weight='bold')

plt.show()

pairs = pd.DataFrame({
    'point_id': np.arange(len(image_points_px)),
    'image_u_px': image_points_px[:, 0],
    'image_v_px': image_points_px[:, 1],
    'ground_x_m': ground_points_m[:, 0],
    'ground_y_m': ground_points_m[:, 1],
})
display(pairs)


In [ ]:
#@title 5. Compute image-to-ground metric homography
if len(image_points_px) < 4:
    raise RuntimeError('Need at least 4 image points.')
if len(image_points_px) != len(ground_points_m):
    raise RuntimeError('image_points_px and ground_points_m must have the same length.')

H_image_to_ground_m, mask = cv2.findHomography(image_points_px, ground_points_m, method=0)
if H_image_to_ground_m is None:
    raise RuntimeError('cv2.findHomography failed.')

H_ground_m_to_image = np.linalg.inv(H_image_to_ground_m)

print('H_image_to_ground_m:')
print(H_image_to_ground_m)
print('Inlier mask:', mask.ravel().tolist() if mask is not None else None)

def project_image_to_ground(points_px, H):
    pts = np.asarray(points_px, dtype=np.float32).reshape(-1, 1, 2)
    out = cv2.perspectiveTransform(pts, H.astype(np.float32)).reshape(-1, 2)
    return out

reprojected_ground = project_image_to_ground(image_points_px, H_image_to_ground_m)
err = np.linalg.norm(reprojected_ground - ground_points_m, axis=1)

check = pairs.copy()
check['projected_ground_x_m'] = reprojected_ground[:, 0]
check['projected_ground_y_m'] = reprojected_ground[:, 1]
check['fit_error_m'] = err

display(check)
print('Mean fit error meters:', float(err.mean()))
print('Max fit error meters:', float(err.max()))


In [ ]:
#@title 6. Create BEV preview from metric homography
# Convert metric ground coordinates to BEV preview pixel coordinates.
# ground x=0..ROAD_WIDTH_M maps to pixel x=0..BEV_PREVIEW_WIDTH_PX.
# ground y=0..ROAD_LENGTH_M maps to pixel y=BEV_PREVIEW_HEIGHT_PX..0 so far road is top.

S_ground_to_bev_px = np.array([
    [BEV_PREVIEW_WIDTH_PX / ROAD_WIDTH_M, 0.0, 0.0],
    [0.0, -BEV_PREVIEW_HEIGHT_PX / ROAD_LENGTH_M, BEV_PREVIEW_HEIGHT_PX],
    [0.0, 0.0, 1.0],
], dtype=np.float64)

H_image_to_bev_px = S_ground_to_bev_px @ H_image_to_ground_m
H_bev_px_to_image = np.linalg.inv(H_image_to_bev_px)

bev_bgr = cv2.warpPerspective(img_bgr, H_image_to_bev_px, (BEV_PREVIEW_WIDTH_PX, BEV_PREVIEW_HEIGHT_PX))
bev_rgb = cv2.cvtColor(bev_bgr, cv2.COLOR_BGR2RGB)

bev_path = NB02B_DIR / 'metric_bev_preview.png'
cv2.imwrite(str(bev_path), bev_bgr)

aspect = BEV_PREVIEW_WIDTH_PX / BEV_PREVIEW_HEIGHT_PX
plt.figure(figsize=(8, 8 / aspect))
plt.imshow(bev_rgb)
plt.title('Metric BEV preview')
plt.axis('off')
plt.show()

print('Saved BEV preview:', bev_path)


In [ ]:
#@title 7. Overlay metric grid and calibration points on BEV preview
bev_debug = bev_rgb.copy()

# Draw grid every 5 meters.
for x_m in np.arange(0, ROAD_WIDTH_M + 1e-6, 5.0):
    x_px = int(round(x_m * BEV_PREVIEW_WIDTH_PX / ROAD_WIDTH_M))
    cv2.line(bev_debug, (x_px, 0), (x_px, BEV_PREVIEW_HEIGHT_PX - 1), (255, 255, 0), 1)
    cv2.putText(bev_debug, f'{x_m:.0f}m', (x_px + 3, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1, cv2.LINE_AA)

for y_m in np.arange(0, ROAD_LENGTH_M + 1e-6, 10.0):
    y_px = int(round(BEV_PREVIEW_HEIGHT_PX - y_m * BEV_PREVIEW_HEIGHT_PX / ROAD_LENGTH_M))
    cv2.line(bev_debug, (0, y_px), (BEV_PREVIEW_WIDTH_PX - 1, y_px), (255, 255, 0), 1)
    cv2.putText(bev_debug, f'{y_m:.0f}m', (5, max(20, y_px - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1, cv2.LINE_AA)

# Project calibration ground points to BEV pixels.
calib_bev = []
for x_m, y_m in ground_points_m:
    x_px = int(round(x_m * BEV_PREVIEW_WIDTH_PX / ROAD_WIDTH_M))
    y_px = int(round(BEV_PREVIEW_HEIGHT_PX - y_m * BEV_PREVIEW_HEIGHT_PX / ROAD_LENGTH_M))
    calib_bev.append((x_px, y_px))

for i, (x_px, y_px) in enumerate(calib_bev):
    cv2.circle(bev_debug, (x_px, y_px), 8, (255, 0, 0), -1)
    cv2.putText(bev_debug, str(i), (x_px + 8, y_px - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2, cv2.LINE_AA)

bev_grid_path = NB02B_DIR / 'metric_bev_preview_grid.png'
cv2.imwrite(str(bev_grid_path), cv2.cvtColor(bev_debug, cv2.COLOR_RGB2BGR))

aspect = BEV_PREVIEW_WIDTH_PX / BEV_PREVIEW_HEIGHT_PX
plt.figure(figsize=(8, 8 / aspect))
plt.imshow(bev_debug)
plt.title('Metric BEV preview with grid')
plt.axis('off')
plt.show()

print('Saved BEV grid preview:', bev_grid_path)


In [ ]:
#@title 8. Reproject metric road rectangle back onto original image
road_corners_m = np.array([
    [0.0, 0.0],
    [ROAD_WIDTH_M, 0.0],
    [ROAD_WIDTH_M, ROAD_LENGTH_M],
    [0.0, ROAD_LENGTH_M],
], dtype=np.float32)

road_corners_img = cv2.perspectiveTransform(road_corners_m.reshape(-1, 1, 2), H_ground_m_to_image.astype(np.float32)).reshape(-1, 2)

debug_img = img_rgb.copy()
poly = road_corners_img.astype(np.int32).reshape(-1, 1, 2)
cv2.polylines(debug_img, [poly], isClosed=True, color=(255, 0, 0), thickness=3)

for i, (u, v) in enumerate(road_corners_img):
    cv2.circle(debug_img, (int(round(u)), int(round(v))), 8, (255, 0, 0), -1)
    cv2.putText(debug_img, str(i), (int(round(u)) + 8, int(round(v)) - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2, cv2.LINE_AA)

# Also draw the actual selected calibration image points.
for i, (u, v) in enumerate(image_points_px):
    cv2.circle(debug_img, (int(round(u)), int(round(v))), 6, (0, 255, 255), -1)
    cv2.putText(debug_img, str(i), (int(round(u)) + 8, int(round(v)) + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2, cv2.LINE_AA)

reproj_path = NB02B_DIR / 'metric_road_rectangle_on_image.png'
cv2.imwrite(str(reproj_path), cv2.cvtColor(debug_img, cv2.COLOR_RGB2BGR))

plt.figure(figsize=(14, 7))
plt.imshow(debug_img)
plt.title('Metric road rectangle reprojected onto image')
plt.axis('off')
plt.show()

print('Saved image reprojection debug:', reproj_path)


In [ ]:
from notebook_local import default_gt_path
#@title 9. Test vehicle ground-point projection using GT annotations if available
GT_PATH_CANDIDATES = [
    PROJECT_DIR / 'GT' / 'M1401_gt_whole.txt',
    default_gt_path('M1401_gt_whole.txt', PROJECT_DIR),
    default_gt_path('M1401_gt_whole.txt', PROJECT_DIR),
]

gt_path = None
for p in GT_PATH_CANDIDATES:
    if p.exists():
        gt_path = p
        break

if gt_path is None:
    print('No M1401_gt_whole.txt found. Skipping GT projection test.')
else:
    print('Using GT annotations:', gt_path)
    cols = ['frame_index', 'target_id', 'bbox_left', 'bbox_top', 'bbox_width', 'bbox_height', 'out_of_view', 'occlusion', 'object_category']
    ann = pd.read_csv(gt_path, header=None, names=cols)
    
    # Use true mapping if available; otherwise infer current original frame from reference index.
    true_mapping_path = WORK_DIR / 'M1401_sample' / 'frame_mapping_true.csv'
    if true_mapping_path.exists():
        mapping = pd.read_csv(true_mapping_path)
        row = mapping[mapping['sample_frame_idx'] == REFERENCE_FRAME_INDEX]
        if len(row) == 0:
            original_frame_index = None
        else:
            original_frame_index = int(row.iloc[0]['original_frame_index'])
    else:
        original_frame_index = None
    
    if original_frame_index is None:
        print('Could not infer original frame index. Skipping GT projection test.')
    else:
        f_ann = ann[ann['frame_index'] == original_frame_index].copy()
        print('Original frame index:', original_frame_index)
        print('GT boxes in reference frame:', len(f_ann))
        
        f_ann['x1'] = f_ann['bbox_left']
        f_ann['y1'] = f_ann['bbox_top']
        f_ann['x2'] = f_ann['bbox_left'] + f_ann['bbox_width']
        f_ann['y2'] = f_ann['bbox_top'] + f_ann['bbox_height']
        f_ann['bbox_cx'] = 0.5 * (f_ann['x1'] + f_ann['x2'])
        f_ann['bbox_ground_y'] = f_ann['y2']
        
        img_pts = f_ann[['bbox_cx', 'bbox_ground_y']].to_numpy(dtype=np.float32)
        ground_pts = project_image_to_ground(img_pts, H_image_to_ground_m)
        f_ann['ground_x_m'] = ground_pts[:, 0]
        f_ann['ground_y_m'] = ground_pts[:, 1]
        
        # Draw on image.
        im_dbg = img_rgb.copy()
        for _, r in f_ann.iterrows():
            x1, y1, x2, y2 = map(int, [r['x1'], r['y1'], r['x2'], r['y2']])
            tid = int(r['target_id'])
            cv2.rectangle(im_dbg, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(im_dbg, str(tid), (x1, max(20, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.circle(im_dbg, (int(round(r['bbox_cx'])), int(round(r['bbox_ground_y']))), 4, (255, 0, 0), -1)
        
        # Draw on BEV grid.
        bev_pts = []
        for _, r in f_ann.iterrows():
            x_px = int(round(r['ground_x_m'] * BEV_PREVIEW_WIDTH_PX / ROAD_WIDTH_M))
            y_px = int(round(BEV_PREVIEW_HEIGHT_PX - r['ground_y_m'] * BEV_PREVIEW_HEIGHT_PX / ROAD_LENGTH_M))
            bev_pts.append((x_px, y_px, int(r['target_id'])))
        
        bev_dbg = bev_debug.copy()
        for x_px, y_px, tid in bev_pts:
            cv2.circle(bev_dbg, (x_px, y_px), 7, (255, 0, 255), -1)
            cv2.putText(bev_dbg, str(tid), (x_px + 8, y_px - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 0, 255), 2, cv2.LINE_AA)
        
        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        axes[0].imshow(im_dbg)
        axes[0].set_title('GT boxes and ground points on image')
        axes[0].axis('off')
        axes[1].imshow(bev_dbg)
        axes[1].set_title('GT ground points in metric BEV')
        axes[1].axis('off')
        plt.show()
        
        gt_proj_path = NB02B_DIR / 'gt_vehicle_projection_test.csv'
        f_ann.to_csv(gt_proj_path, index=False)
        print('Saved GT projection test CSV:', gt_proj_path)


In [ ]:
#@title 10. Save metric homography calibration JSON
calibration = {
    'version': '02b_visual_metric_homography_v1',
    'reference_frame_index': int(REFERENCE_FRAME_INDEX),
    'reference_frame_path': str(ref_path),
    'reference_frame_name': ref_path.name,
    'image_width_px': int(IMG_W),
    'image_height_px': int(IMG_H),
    'road_width_m': float(ROAD_WIDTH_M),
    'road_length_m': float(ROAD_LENGTH_M),
    'bev_preview_width_px': int(BEV_PREVIEW_WIDTH_PX),
    'bev_preview_height_px': int(BEV_PREVIEW_HEIGHT_PX),
    'image_points_px': image_points_px.astype(float).tolist(),
    'ground_points_m': ground_points_m.astype(float).tolist(),
    'H_image_to_ground_m': H_image_to_ground_m.astype(float).tolist(),
    'H_ground_m_to_image': H_ground_m_to_image.astype(float).tolist(),
    'H_image_to_bev_px': H_image_to_bev_px.astype(float).tolist(),
    'coordinate_system': {
        'x': 'meters across road from left boundary to right boundary in calibrated BEV',
        'y': 'meters along road from near boundary to far boundary in calibrated BEV',
        'z': 'meters above road plane',
    },
    'fit_error_m': {
        'mean': float(err.mean()),
        'max': float(err.max()),
        'per_point': err.astype(float).tolist(),
    },
    'outputs': {
        'metric_bev_preview': str(bev_path),
        'metric_bev_preview_grid': str(bev_grid_path),
        'metric_road_rectangle_on_image': str(reproj_path),
    },
}

calibration_path = NB02B_DIR / 'metric_homography_calibration.json'
with open(calibration_path, 'w') as f:
    json.dump(calibration, f, indent=2)

# Also save a compatibility file with the old name pattern if later notebooks search *homography*.json.
compat_path = NB02B_DIR / 'frame_00000_metric_homography.json'
with open(compat_path, 'w') as f:
    json.dump(calibration, f, indent=2)

print('Saved calibration JSON:', calibration_path)
print('Saved compatibility JSON:', compat_path)


In [ ]:
#@title 11. Summary
print('Notebook 02B complete.')
print('Use this calibration in later notebooks:')
print(NB02B_DIR / 'metric_homography_calibration.json')
print('Key matrix: H_image_to_ground_m')
print('Road width meters:', ROAD_WIDTH_M)
print('Road length meters:', ROAD_LENGTH_M)
